# COVID context
(breakups, alternatives and CLalt)

**Inputs:** `hcmst2017_person_period.csv` (02), `model_results.csv` (04), original HCMST file in `data/raw/`

**Purpose:** to give descriptive context for the drop in breakups after 2020 and discuss it in terms of CLalt (comparison level of alternatives). This notebook does not add new models

The COVID questions were asked in 2020, so they cannot predict the 2017-2020 breakups. We use only closed-ended questions, for description.

## 1. Setting up

Load `pandas` and set the file paths used in this notebook.

In [1]:
import numpy as np
import pandas as pd

In [2]:
RAW = "../data/raw/HCMST_2017_to_2022_small_public_version_2.2.dta"
OUT = "../output/tables/"

## 2. Breakup rate in each survey gap

Share of couples who broke up in each gap, from the person-period file (one row per couple per gap). 

The 2020-2022 gap is shorter, so we also show an approximate yearly rate, dividing by gap length (about 3 years for 2017–2020 and 2 years for 2020-2022).

In [3]:
pp = pd.read_csv("../data/processed/hcmst2017_person_period.csv")

gap = pp.groupby("period")["broke_up"].agg(rows="size", breakups="sum", rate="mean").reset_index()
gap["rate_pct"] = (gap["rate"] * 100).round(1)
gap["approx_years"] = gap["period"].map({"2017-2020": 3, "2020-2022": 2})
gap["approx_rate_per_year_pct"] = (gap["rate_pct"] / gap["approx_years"]).round(1)

gap.to_csv(OUT + "covid_context_breakups_by_gap.csv", index=False)
gap

,period,rows,breakups,rate,rate_pct,approx_years,approx_rate_per_year_pct
0,2017-2020,1675,183,0.109254,10.9,3.0,3.6
1,2017-2022,89,15,0.168539,16.9,NaN,NaN
2,2020-2022,1119,20,0.017873,1.8,2.0,0.9


## 3. Is the drop still there after controls?

Pull the 2020-2022 gap term from the models saved in notebook 04. Model 4 already controls for income, race, who earned more, marriage, relationship length, age, same-sex couple and children, so an odds ratio below 1 means breakups were less likely in 2020-2022 than in 2017-2020, comparing similar couples.

In [4]:
model_results = pd.read_csv(OUT + "model_results.csv")

period_rows = model_results[model_results["term"].str.contains("period")]
period_rows.to_csv(OUT + "covid_context_period_effect.csv", index=False)

period_m4 = period_rows[period_rows["model"] == "Model 4"]
period_m4

,term,model,coef,odds_ratio,p_value,ci_lower,ci_upper
31,C(period)[T.2017-2022],Model 4,0.502,1.651,0.205,-0.274,1.277
32,C(period)[T.2020-2022],Model 4,-1.460,0.232,0.000,-1.996,-0.924


## 4. Loading the COVID questions

Load the original HCMST file and check that the three closed-ended COVID questions are in it:
- `w2_corona_effect_dating_combo`: the effect of the pandemic on meeting people in 2020 (singles)
- `w3_coronavirus_effect_dating`: the same question in 2022 (singles)
- `w2_coronavirus_effect_combo`: the effect of the pandemic on the relationship in 2020 (couples)

In [5]:
raw = pd.read_stata(RAW, convert_categoricals=True)

covid_vars = ["w2_corona_effect_dating_combo", "w3_coronavirus_effect_dating", "w2_coronavirus_effect_combo"]
print("Raw:", raw.shape)
print("Missing from file:", [c for c in covid_vars if c not in raw.columns])
print("COVID-related columns:", [c for c in raw.columns if "corona" in c.lower()])

C:\Users\vivid\AppData\Local\Temp\ipykernel_10040\2465778202.py:1: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  raw = pd.read_stata(RAW, convert_categoricals=True)


Raw: (3510, 725)
Missing from file: []
COVID-related columns: ['w3_coronavirus_effect_combo', 'w3_corona_effect_combo_reversed', 'w3_how_many_corona', 'w3_corona_vaccine', 'w3_partner_corona_vaccine', 'w3_corona_test', 'w3_partcorona_test', 'w3_coronavirus_effect_dating', 'w3_corona_longing', 'w2_coronavirus_effect_combo', 'w2_corona_effect_combo_reversed', 'w2_how_many_corona_combo', 'w2_corona_effect_dating_combo', 'w2_corona_longing_combo', 'w2_corona_test', 'w2_partcorona_test', 'w2_partcorona_test_new']


## 5. What respondents said about COVID

The count and percentage for each answer. 

The dating question is only asked for singles, so it describes the dating market people would be leaving into, not the alternatives of people who are partnered.
The relationship question is limited to couples still with their 2017 partner in 2020, taken from the person-period file (no breakup in 2017-2020).

In [6]:
def tab(series, label):
    counts = series.value_counts(dropna=True)
    out = pd.DataFrame({"answer": counts.index.astype(str), "n": counts.values,
                        "pct": (counts.values / counts.sum() * 100).round(1)})
    out.insert(0, "question", label)
    return out

# Couples still with their 2017 partner in 2020
still2020 = pp.loc[(pp["period"] == "2017-2020") & (pp["broke_up"] == 0), "caseid_new"]
couples = raw[raw["caseid_new"].isin(still2020)]

covid_summary = pd.concat([
    tab(raw["w2_corona_effect_dating_combo"], "Singles 2020: effect on meeting people"),
    tab(raw["w3_coronavirus_effect_dating"], "Singles 2022: effect on meeting people"),
    tab(couples["w2_coronavirus_effect_combo"], "Couples still together in 2020: effect on relationship"),
], ignore_index=True)

covid_summary.to_csv(OUT + "covid_context_summary.csv", index=False)
covid_summary

,question,answer,n,pct
0,Singles 2020: effect on meeting people,It is HARDER to meet people face-to-face Now t...,245,52.4
1,Singles 2020: effect on meeting people,No change,219,46.8
2,Singles 2020: effect on meeting people,It is EASIER to meet people face-to-face Now t...,4,0.9
3,Singles 2022: effect on meeting people,No change,310,64.9
4,Singles 2022: effect on meeting people,It is HARDER to meet people face-to-face Now t...,158,33.1
5,Singles 2022: effect on meeting people,It is EASIER to meet people face-to-face Now t...,10,2.1
6,Couples still together in 2020: effect on rela...,no change,1150,77.4
7,Couples still together in 2020: effect on rela...,better rel than before,257,17.3
8,Couples still together in 2020: effect on rela...,worse rel than before,78,5.3


## 6. Final check

In [12]:
assert {"2017-2020", "2020-2022"}.issubset(set(gap["period"])), "Expected the 2017-2020 and 2020-2022 gaps"

row_2020_2022 = period_m4[period_m4["term"].str.contains("2020-2022")]
assert len(row_2020_2022) == 1 and row_2020_2022["odds_ratio"].iloc[0] < 1, "Breakup odds should be lower in 2020-2022"

assert covid_summary["question"].nunique() == 3, "Expected three COVID question tables"

print("All checks passed ✓")

All checks passed ✓
